In [ ]:
# %env Q_ALCHEMY_API_KEY=<api_key>

env: Q_ALCHEMY_API_KEY=qal_5NyfFhLs_3yPYAAbTxBDFK8DSUk9KoPFf


# One molecule, one Quantum I/O workflow

Prepare the **12-qubit lithium hydride (LiH/STO-3G) CISD state**, acquire observables, and reconstruct a QTucker state using the public Q-Alchemy SDK.

This replaces the preparation, acquisition, and reconstruction stages of the three `qtucker-examples/ibm_experiment` notebooks with **one service request**. The included sparse target is the same normalized LiH state generated by the first notebook (35 amplitudes). Chemistry is already computed; you do not need PySCF or private QTucker packages locally.

Choose `ideal`, `noisy`, or `qpu` below. Ideal is the default. The noisy case imports a saved IBM device model and requires no IBM account. The QPU case executes on the IBM backend you select.

**Requirements:** SDK 0.5.0 and deployed Quantum I/O PineXQ 0.8.4 backed by core 0.12.2. Install this SDK revision into your notebook kernel before those versions are published. Set `Q_ALCHEMY_API_KEY` (or `PINEXQ_API_KEY`) in your environment or a local `.env`. For QPU mode, also set `IBM_QUANTUM_TOKEN` and optionally `IBM_QUANTUM_INSTANCE`.

For a released SDK: `pip install "q-alchemy-sdk-py[examples]>=0.5.0"`. This extra supplies the public Qiskit, Aer, IBM runtime, and notebook dependencies. Run from the SDK root or its `examples` folder.

In [2]:
import os
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from q_alchemy import (
    ExecutionPlan, IBMQuantumCredentials, MeasurementPlan,
    QuantumExperiment, QuantumIOParams, QuantumIOService, Runtime, State,
)

from q_alchemy.quantum_io import CircuitCompressionConfig

load_dotenv()  # Reads credentials without embedding them in the notebook.


False

## Choose the experiment

The defaults match the saved LiH settings in the original IBM experiment: iterative preparation with a 0.001 estimated loss budget, six blocks of **two qubits**, rank one, **95 training observables**, four restarts, and 600 AdamW fitting epochs. The preparation, observable strategy, and fitting settings come from `lih.observable_strategy.json` and `lih.fit_config.json`; the fitting controls are explicit below. Start with `ideal` to check the workflow, then change only `MODE`.

The original observable family is retained as training data, without a validation split. `INDEPENDENT_VALIDATION=True` adds a separate family for held-out checking; it changes the experiment and increases acquisition work. Increasing `RANK`, `EPOCHS`, or `RESTARTS` may improve fitting, at additional cost. Acquisition mode and optional compression remain independent choices; use `ENABLE_COMPRESSION=False` for the original uncompressed baseline.


In [3]:
MODE = "ideal"  # "ideal", "noisy", or "qpu"
IBM_BACKEND = ""  # Required in QPU mode: use a device available to your account.
SHOTS = 4096
SIMULATOR_METHOD = "statevector"  # Or "matrix_product_state" / "automatic" for noisy mode.
BLOCK_SIZE = 2
RANK = 1
EPOCHS = 600
RESTARTS = 4
INDEPENDENT_VALIDATION = False
ENABLE_COMPRESSION = True
OUTPUT_DIR = Path("lih_reconstruction_outputs")

if MODE not in {"ideal", "noisy", "qpu"}:
    raise ValueError("MODE must be ideal, noisy, or qpu")

if SIMULATOR_METHOD not in {"matrix_product_state", "statevector", "automatic"}:
    raise ValueError("SIMULATOR_METHOD must be matrix_product_state, statevector, or automatic")


## Load the sparse molecular target

LiH has six spatial orbitals, represented by twelve spin-orbital qubits, with geometry `Li 0 0 0; H 0 0 1.60` in Ångström. The included file preserves the original experiment's determinant-to-index convention. Each index refers to the corresponding integer basis state in the SDK; no additional bit reversal is applied.

Only the 35 sparse amplitudes are loaded and transmitted. No `2**12` dense target vector is created.


In [4]:
data_dir = Path("data") if Path("data/lih_sto3g_cisd.npz").exists() else Path("examples/data")
with np.load(data_dir / "lih_sto3g_cisd.npz", allow_pickle=False) as saved:
    indices = saved["indices"].copy()
    amplitudes = saved["amplitudes"].copy()
    num_qubits = int(saved["num_qubits"])

assert num_qubits == 12
assert np.isclose(np.vdot(amplitudes, amplitudes).real, 1.0)
target = State.sparse(num_qubits=num_qubits, indices=indices, amplitudes=amplitudes)
print(f"LiH/STO-3G: {num_qubits} qubits, {len(indices)} nonzero amplitudes")


LiH/STO-3G: 12 qubits, 35 nonzero amplitudes


## Select how observables are acquired

- **Ideal:** direct sparse simulation evaluates expectations without shot noise or measurement circuits. `SHOTS` is not sampled on this path.
- **Noisy:** import `FakeAlmadenV2`, IBM's historical 20-qubit device snapshot, and retain its calibration errors and connectivity only on physical qubits 0–11. These form a connected 12-qubit device model. Aer runs on the service with finite shots, using `SIMULATOR_METHOD`, and the compilation target is explicitly limited to 12 qubits. This is a simulation based on historical calibration, not current live device calibration.
- **QPU:** supply your IBM credentials separately and select a backend with at least 12 qubits. Physical compilation and observable layout mapping are handled by Quantum I/O. Running the submission cell in this mode submits real hardware work.

The SDK sends only portable settings; it does not select preparation candidates or fit the model locally.

`statevector` is selected for this smaller experiment: a double-precision 12-qubit state vector is 64 KiB, plus temporary buffers and any parallel noisy trajectories. `matrix_product_state` is also available and can keep memory small for states with limited entanglement. `automatic` lets Aer select a method; it is not a guarantee of lower cost.

For a quick workflow check, reduce `SHOTS` (for example to 256 or 512), `EPOCHS`, and `RESTARTS`; this reduces statistical precision or fitting effort. Ideal mode remains the least expensive acquisition option, using direct sparse expectations. The smaller target does not change the fitting controls: four restarts of 600 epochs still request substantial work.


In [5]:
credentials = None

if MODE == "ideal":
    acquisition = Runtime.qalchemy_sparse()

elif MODE == "noisy":
    from qiskit_ibm_runtime.fake_provider import FakeAlmadenV2
    from qiskit_aer.noise import NoiseModel

    device = FakeAlmadenV2()
    noise_model = NoiseModel.from_backend(device)
    # Keep the connected first 12 physical qubits, without relabeling them.
    noise_data = noise_model.to_dict(serializable=True)
    noise_data["errors"] = [
        error for error in noise_data["errors"]
        if all(q < num_qubits for qubits in error.get("gate_qubits", []) for q in qubits)
    ]
    coupling_map = [
        [left, right] for left, right in device.coupling_map.get_edges()
        if left < num_qubits and right < num_qubits
    ]
    acquisition = Runtime.resource(
        "noisy-backend-simulator",
        provider="aer",
        backend="aer_simulator",
        backend_options={
            "n_qubits": num_qubits,
            "method": SIMULATOR_METHOD,
            "noise_model": {
                "format": "qiskit-aer-noise-model-v1",
                "data": noise_data,
            },
            "basis_gates": noise_model.basis_gates,
            "transpile": True,
            "transpile_options": {
                "basis_gates": noise_model.basis_gates,
                "coupling_map": coupling_map,
                "initial_layout": list(range(num_qubits)),
                "optimization_level": 1,
                "seed_transpiler": 123,
            },
            "estimator_options": {"seed_simulator": 123},
        },
    )

else:
    if not IBM_BACKEND.strip():
        raise ValueError("Set IBM_BACKEND to an accessible IBM device with at least 12 qubits")
    credentials = IBMQuantumCredentials(
        token=os.environ["IBM_QUANTUM_TOKEN"],
        instance=os.getenv("IBM_QUANTUM_INSTANCE") or None,
    )
    acquisition = Runtime.resource(
        "quantum-backend", provider="ibm", backend=IBM_BACKEND,
        execution_options={"transpile": True, "transpile_options": {"optimization_level": 1}},
    )

print(f"Acquisition mode: {MODE}")


Acquisition mode: ideal


## Describe preparation and reconstruction

There is no evolution circuit: we reconstruct the prepared molecular state, as in the original notebooks. Observable names, measurement circuits, acquisition, and fitting are generated on the service.

`ENABLE_COMPRESSION=True` enables exact compression of the logical preparation circuit before acquisition in all three modes. Compression preserves its output from `|0...0>`; it does not repair preparation error against the molecular target. Set the flag to `False` to compare without this additional stage. The report summary and saved report include compression status and before/after circuit metrics.

The fitted QTucker model stays implicit. Request amplitudes only on the input target's support and compare the full normalized model with that target. The exported subset is **not renormalized**, and model amplitudes outside the exported support are not assumed zero.

In [6]:
experiment = QuantumExperiment(
    target=target,
    measurement_plan=MeasurementPlan.qtucker_reconstruction(
        validation=INDEPENDENT_VALIDATION,
        blocks=None, block_size=BLOCK_SIZE,
        rank=RANK, ranks=None,
        within_block_mode="all_paulis",
        cross_block_mode="all_pairs",
        cross_block_max_locality=4,
        extra_edges=None,
        rank_reduce_within_blocks=False,
        rank_reduce_cross_blocks=True,
        rank_observable_floor=1,
        rank_cross_block_floor=1,
        paulis="XYZ",
    ),
    metadata={"name": "lih-sto3g-cisd-12-qubits", "acquisition_mode": MODE},
)
plan = ExecutionPlan(
    preparation_method="iterative_tucker",
    preparation_options={
        "basis_gates": ["u", "cx"],
        "fallback": False,
        "factors_size": 0,
        "max_fidelity_loss": 1e-3,
        "max_iterations": 1,
    },
    circuit_compression=CircuitCompressionConfig(enabled=ENABLE_COMPRESSION),
    acquisition=acquisition,
    estimator=Runtime.qtucker(
        blocks=None, block_size=BLOCK_SIZE, rank=RANK, ranks=None,
        n_restarts=RESTARTS, device=None,
        # Match lih.fit_config.json, including controls otherwise left to defaults.
        # The estimator uses complex128, as in the original LiH fit.
        fit_config={
            "optimizer": "adamw", "epochs": EPOCHS,
            "learning_rate": 0.03, "batch_size": None, "stderr_floor": 0.01,
            "weight_decay": 0.0, "grad_clip_norm": 10.0,
            "gauge_fix_every": 5, "validation_every": 10,
            "loss_backend": "auto", "loss_compile_mode": "auto",
            "loss_chunk_size": None, "loss_max_chunk_bytes": None,
            "empty_device_cache_every": None,
            "release_memory_on_return": True, "trim_cpu_memory_on_return": True,
            "lbfgs_max_iter": 20, "lbfgs_history_size": 50,
            "lbfgs_line_search_fn": "strong_wolfe",
            "seed": 123, "verbose": True,
        },
    ),
    estimation_output={"support": "target", "target_fidelity": True},
    shots=SHOTS,
)


## Submit one job

This cell calls the deployed service in every mode. Preparation and fit settings are passed through to their owning libraries. Do not rerun it merely to redisplay results: the saved report below can be loaded independently.

In [ ]:
with QuantumIOService(
    QuantumIOParams(job_completion_timeout_sec=600, remove_data=True), ibm_credentials=credentials,
) as service:
    report = service.run(experiment, plan).result()

print(report.format_summary())


STATE PREPARATION (target vs prepared state)
--------------------------------------------
Target qubits:             12
Method:                    iterative_tucker
Claimed fidelity loss (initializer): 0.0127775
Preparation simulation:  q-alchemy-simulator:SparseAerBackend (exact-within-floating-point)
Target-to-prepared fidelity: 0.987222
Preparation infidelity: 0.0127775
Preparation circuit:      CX count=9, two-qubit count=9, depth=7, size=27, operations=cx=9, u=18

FULL EXPERIMENT CIRCUIT (P + U)
-------------------------------
Evolution U present:       no
Complete circuit:         CX count=8, two-qubit count=8, depth=7, size=23, operations=cx=8, u=15

CIRCUIT COMPRESSION
  Attempted: True
  Applied: True
  Changed: True
  Circuit used: compressor-output
  Result: compressor output selected; circuit changed
  Equivalence: reachable_subspace
  1Q operations: 18 -> 15
  2Q operations: 9 -> 8
  Depth: 7 -> 7
  Accepted regions: 1

FINAL OUTPUT COMPARISON (ideal/reference P + U vs acqu

## Save and inspect the reconstruction

Inspect preparation `found`, its estimated loss, measured preflight fidelity (when available), and report warnings. The preparation loss estimate and reconstruction fidelity describe different stages. A failed search or poor reconstruction should not be treated as a successful molecular preparation.

`target_fidelity` compares the fitted normalized **pure-state surrogate** with the original CISD target. On noisy simulation or hardware it does not prove recovery of the full mixed state. With training-only data there is no independent held-out check, and matching these observables does not guarantee high target fidelity.

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "report.json").write_text(report.to_json(indent=2), encoding="utf-8")
(OUTPUT_DIR / "experiment.json").write_text(experiment.to_json(indent=2), encoding="utf-8")
(OUTPUT_DIR / "execution_plan.json").write_text(plan.to_json(indent=2), encoding="utf-8")

if report.estimate is None or report.estimate.sparse_state is None:
    raise RuntimeError("No sparse reconstruction was returned; inspect the saved report and warnings")

estimate = report.estimate
estimate.sparse_state.save_npz(OUTPUT_DIR / "reconstructed.npz")
print("Target-to-estimated fidelity:", estimate.target_fidelity)
print("Exported amplitudes:", len(estimate.sparse_state.indices))
print("Probability on exported support:", estimate.sparse_state.metadata.get("returned_probability"))
print("Saved results to:", OUTPUT_DIR.resolve())


Target-to-estimated fidelity: 0.9741897327761014
Exported amplitudes: 35
Probability on exported support: 0.9999788620277876
Saved results to: /media/israel/DC/Data Cybernetics/github/q-alchemy-sdk-py/examples/lih_reconstruction_outputs


To inspect saved results without another service call:

```python
from q_alchemy import ExperimentReport
saved_report = ExperimentReport.from_json((OUTPUT_DIR / "report.json").read_text())
print(saved_report.format_summary())
```

The NPZ contains decimal basis indices, complex amplitudes, and JSON metadata, and loads with `np.load(path, allow_pickle=False)`. The report retains measured observables and fit diagnostics. See `examples/data/README.md` for target provenance.